In [1]:
# Імпорт необхідних бібліотек
import pandas as pd
import urllib.request
import os
from datetime import datetime
import time

print("Бібліотеки імпортовані успішно!")

Бібліотеки імпортовані успішно!


In [2]:
# Функція для завантаження VHI даних
def download_vhi_data(province_id, year1=1981, year2=2024):
    """
    Завантажує VHI дані для вказаної області
    
    Parameters:
    province_id (int): ID області (1-27, НЕ 0)
    year1 (int): Початковий рік
    year2 (int): Кінцевий рік
    """
    # Створюємо теку для даних якщо не існує
    if not os.path.exists('vhi_data'):
        os.makedirs('vhi_data')
    
    # Формуємо URL
    url = f'https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1={year1}&year2={year2}&type=Mean'
    
    # Генеруємо ім'я файлу з датою та часом
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'vhi_data/vhi_province_{province_id}_{timestamp}.csv'
    
    # Перевіряємо чи файл вже існує (для запобігання повторного завантаження)
    existing_files = [f for f in os.listdir('vhi_data') if f.startswith(f'vhi_province_{province_id}_')]
    
    if existing_files:
        print(f"Область {province_id}: файл вже існує ({existing_files[0]}), пропускаємо")
        return f'vhi_data/{existing_files[0]}'
    
    try:
        # Завантажуємо файл
        print(f"Завантаження даних для області {province_id}...")
        urllib.request.urlretrieve(url, filename)
        print(f"✓ Область {province_id}: дані завантажено ({filename})")
        time.sleep(1)  # Пауза щоб не перевантажувати сервер
        return filename
    except Exception as e:
        print(f"✗ Помилка завантаження для області {province_id}: {e}")
        return None

# Тест функції
test_file = download_vhi_data(1)
print(f"\nТестовий файл: {test_file}")

Завантаження даних для області 1...
✓ Область 1: дані завантажено (vhi_data/vhi_province_1_20260218_121125.csv)

Тестовий файл: vhi_data/vhi_province_1_20260218_121125.csv


In [3]:
# Завантажуємо дані для всіх областей (1-27, пропускаємо 0)
print("Початок завантаження даних для всіх областей України...\n")

downloaded_files = {}
for province_id in range(1, 28):  # 27 областей
    filename = download_vhi_data(province_id)
    if filename:
        downloaded_files[province_id] = filename

print(f"\n✓ Завантажено файлів: {len(downloaded_files)}")

Початок завантаження даних для всіх областей України...

Область 1: файл вже існує (vhi_province_1_20260218_121125.csv), пропускаємо
Завантаження даних для області 2...
✓ Область 2: дані завантажено (vhi_data/vhi_province_2_20260218_121146.csv)
Завантаження даних для області 3...
✓ Область 3: дані завантажено (vhi_data/vhi_province_3_20260218_121149.csv)
Завантаження даних для області 4...
✓ Область 4: дані завантажено (vhi_data/vhi_province_4_20260218_121152.csv)
Завантаження даних для області 5...
✓ Область 5: дані завантажено (vhi_data/vhi_province_5_20260218_121154.csv)
Завантаження даних для області 6...
✓ Область 6: дані завантажено (vhi_data/vhi_province_6_20260218_121156.csv)
Завантаження даних для області 7...
✓ Область 7: дані завантажено (vhi_data/vhi_province_7_20260218_121158.csv)
Завантаження даних для області 8...
✓ Область 8: дані завантажено (vhi_data/vhi_province_8_20260218_121200.csv)
Завантаження даних для області 9...
✓ Область 9: дані завантажено (vhi_data/vhi_pro

In [7]:
# Спочатку подивимось на структуру файлу
with open(downloaded_files[1], 'r') as f:
    for i, line in enumerate(f):
        print(f"Рядок {i}: {repr(line)}")
        if i > 5:
            break

Рядок 0: "Mean data for UKR  Province= 1: Cherkasy,  from 1981 to 2024, weekly; version='GC_current'<br>for cropland area only<br>\n"
Рядок 1: 'year,week, SMN,SMT,VCI,TCI, VHI<br>\n'
Рядок 2: '<tt><pre>1982, 1, 0.053,260.31, 45.01, 39.46, 42.23,\n'
Рядок 3: '1982, 2, 0.054,262.29, 46.83, 31.75, 39.29,\n'
Рядок 4: '1982, 3, 0.055,263.82, 48.13, 27.24, 37.68,\n'
Рядок 5: '1982, 4, 0.053,265.33, 46.09, 23.91, 35.00,\n'
Рядок 6: '1982, 5, 0.050,265.66, 41.46, 26.65, 34.06,\n'


In [8]:
def read_and_clean_vhi(filename, province_id):
    """
    Читає CSV файл та очищує дані від HTML-тегів та зайвих символів
    """
    rows = []
    
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            # Прибираємо HTML теги та пробіли
            line = line.replace('<br>', '').replace('<tt>', '').replace('<pre>', '').strip()
            
            # Пропускаємо рядки що не є даними (заголовки, пусті рядки)
            if not line or line.startswith('year') or line.startswith('Mean') or line.startswith('for'):
                continue
            
            # Розбиваємо по комі та прибираємо пусті елементи (остання кома дає пустий елемент)
            parts = [p.strip() for p in line.split(',')]
            parts = [p for p in parts if p != '']
            
            if len(parts) == 7:
                rows.append(parts)
    
    # Створюємо DataFrame
    df = pd.DataFrame(rows, columns=['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'])
    
    # Конвертуємо типи
    for col in ['year', 'week']:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
    for col in ['SMN', 'SMT', 'VCI', 'TCI', 'VHI']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Видаляємо рядки де VHI = -1 (відсутні дані)
    df = df[df['VHI'] != -1]
    
    # Видаляємо рядки з пропусками
    df = df.dropna()
    
    # Додаємо ID області
    df['province_id'] = province_id
    
    return df

# Тест
test_df = read_and_clean_vhi(downloaded_files[1], 1)
print("Приклад даних:")
print(test_df.head(10))
print(f"\nРозмір: {test_df.shape}")
print(f"Стовпці: {list(test_df.columns)}")
print(f"\nТипи даних:\n{test_df.dtypes}")

Приклад даних:
   year  week    SMN     SMT    VCI    TCI    VHI  province_id
0  1982     1  0.053  260.31  45.01  39.46  42.23            1
1  1982     2  0.054  262.29  46.83  31.75  39.29            1
2  1982     3  0.055  263.82  48.13  27.24  37.68            1
3  1982     4  0.053  265.33  46.09  23.91  35.00            1
4  1982     5  0.050  265.66  41.46  26.65  34.06            1
5  1982     6  0.048  266.55  36.56  29.46  33.01            1
6  1982     7  0.048  267.84  32.17  31.14  31.65            1
7  1982     8  0.050  269.30  30.30  32.50  31.40            1
8  1982     9  0.052  270.75  28.23  35.22  31.73            1
9  1982    10  0.056  272.73  25.25  37.63  31.44            1

Розмір: (2186, 8)
Стовпці: ['year', 'week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'province_id']

Типи даних:
year             Int64
week             Int64
SMN            float64
SMT            float64
VCI            float64
TCI            float64
VHI            float64
province_id      int64


In [10]:
# Дивимось заголовки всіх файлів
for province_id, filename in sorted(downloaded_files.items()):
    with open(filename, 'r') as f:
        first_line = f.readline().strip()
        # Витягуємо тільки назву області
        name_part = first_line.split('Province=')[1].split(',')[0].strip()
        print(f"Province {province_id:2d}: {name_part}")

Province  1: 1: Cherkasy
Province  2: 2: Chernihiv
Province  3: 3: Chernivtsi
Province  4: 4: Crimea
Province  5: 5: Dnipropetrovs'k
Province  6: 6: Donets'k
Province  7: 7: Ivano-Frankivs'k
Province  8: 8: Kharkiv
Province  9: 9: Kherson
Province 10: 10: Khmel'nyts'kyy
Province 11: 11: Kiev
Province 12: 12: Kiev City
Province 13: 13: Kirovohrad
Province 14: 14: Luhans'k
Province 15: 15: L'viv
Province 16: 16: Mykolayiv
Province 17: 17: Odessa
Province 18: 18: Poltava
Province 19: 19: Rivne
Province 20: 20: Sevastopol'
Province 21: 21: Sumy
Province 22: 22: Ternopil'
Province 23: 23: Transcarpathia
Province 24: 24: Vinnytsya
Province 25: 25: Volyn
Province 26: 26: Zaporizhzhya
Province 27: 27: Zhytomyr


In [11]:
PROVINCE_MAPPING = {
    1:  {'ua_name': 'Черкаська',          'ua_id': 24},  # Cherkasy
    2:  {'ua_name': 'Чернігівська',        'ua_id': 25},  # Chernihiv
    3:  {'ua_name': 'Чернівецька',         'ua_id': 26},  # Chernivtsi
    4:  {'ua_name': 'АР Крим',             'ua_id': 27},  # Crimea
    5:  {'ua_name': 'Дніпропетровська',    'ua_id': 4},   # Dnipropetrovsk
    6:  {'ua_name': 'Донецька',            'ua_id': 5},   # Donetsk
    7:  {'ua_name': 'Івано-Франківська',   'ua_id': 6},   # Ivano-Frankivsk
    8:  {'ua_name': 'Харківська',          'ua_id': 22},  # Kharkiv
    9:  {'ua_name': 'Херсонська',          'ua_id': 23},  # Kherson
    10: {'ua_name': 'Хмельницька',         'ua_id': 21},  # Khmelnytskyi
    11: {'ua_name': 'Київська',            'ua_id': 9},   # Kiev
    12: {'ua_name': 'м. Київ',             'ua_id': 10},  # Kiev City
    13: {'ua_name': 'Кіровоградська',      'ua_id': 11},  # Kirovohrad
    14: {'ua_name': 'Луганська',           'ua_id': 12},  # Luhansk
    15: {'ua_name': 'Львівська',           'ua_id': 13},  # Lviv
    16: {'ua_name': 'Миколаївська',        'ua_id': 14},  # Mykolaiv
    17: {'ua_name': 'Одеська',             'ua_id': 15},  # Odessa
    18: {'ua_name': 'Полтавська',          'ua_id': 16},  # Poltava
    19: {'ua_name': 'Рівненська',          'ua_id': 17},  # Rivne
    20: {'ua_name': 'Севастополь',         'ua_id': 20},  # Sevastopol
    21: {'ua_name': 'Сумська',             'ua_id': 18},  # Sumy
    22: {'ua_name': 'Тернопільська',       'ua_id': 19},  # Ternopil
    23: {'ua_name': 'Закарпатська',        'ua_id': 7},   # Transcarpathia
    24: {'ua_name': 'Вінницька',           'ua_id': 1},   # Vinnytsia
    25: {'ua_name': 'Волинська',           'ua_id': 2},   # Volyn
    26: {'ua_name': 'Запорізька',          'ua_id': 8},   # Zaporizhzhia
    27: {'ua_name': 'Житомирська',         'ua_id': 3},   # Zhytomyr
}

In [13]:
def load_all_provinces(downloaded_files):
    all_dfs = []
    
    for province_id, filename in downloaded_files.items():
        df = read_and_clean_vhi(filename, province_id)
        
        mapping = PROVINCE_MAPPING.get(province_id, {})
        df['province_name'] = mapping.get('ua_name', f'Область {province_id}')
        df['ua_id'] = mapping.get('ua_id', province_id)
        
        all_dfs.append(df)
    
    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df = combined_df.sort_values(['ua_id', 'year', 'week']).reset_index(drop=True)
    
    return combined_df

In [14]:
vhi_df = load_all_provinces(downloaded_files)

print(f"Загальний розмір датасету: {vhi_df.shape}")
print(f"\nУнікальні області (відсортовані за ua_id):")
print(vhi_df[['province_id', 'ua_id', 'province_name']].drop_duplicates().sort_values('ua_id').to_string(index=False))

Загальний розмір датасету: (59022, 10)

Унікальні області (відсортовані за ua_id):
 province_id  ua_id     province_name
          24      1         Вінницька
          25      2         Волинська
          27      3       Житомирська
           5      4  Дніпропетровська
           6      5          Донецька
           7      6 Івано-Франківська
          23      7      Закарпатська
          26      8        Запорізька
          11      9          Київська
          12     10           м. Київ
          13     11    Кіровоградська
          14     12         Луганська
          15     13         Львівська
          16     14      Миколаївська
          17     15           Одеська
          18     16        Полтавська
          19     17        Рівненська
          21     18           Сумська
          22     19     Тернопільська
          20     20       Севастополь
          10     21       Хмельницька
           8     22        Харківська
           9     23        Херсонська
     

In [16]:
# Функція 1: Ряд VHI для області за вказаний рік
def vhi_by_region_and_year(df, ua_id, year):
    """
    Повертає VHI для вказаної області та року
    """
    result = df[(df['ua_id'] == ua_id) & (df['year'] == year)][['year', 'week', 'VHI', 'province_name']]
    province = df[df['ua_id'] == ua_id]['province_name'].values[0]
    print(f"VHI для області '{province}' за {year} рік:")
    print(result.to_string(index=False))
    return result

# Тест
vhi_by_region_and_year(vhi_df, ua_id=1, year=2010);

VHI для області 'Вінницька' за 2010 рік:
 year  week   VHI province_name
 2010     1 52.04     Вінницька
 2010     2 51.05     Вінницька
 2010     3 52.37     Вінницька
 2010     4 52.69     Вінницька
 2010     5 53.16     Вінницька
 2010     6 53.20     Вінницька
 2010     7 53.00     Вінницька
 2010     8 53.48     Вінницька
 2010     9 53.69     Вінницька
 2010    10 51.78     Вінницька
 2010    11 47.39     Вінницька
 2010    12 45.75     Вінницька
 2010    13 43.61     Вінницька
 2010    14 39.46     Вінницька
 2010    15 38.12     Вінницька
 2010    16 41.33     Вінницька
 2010    17 44.79     Вінницька
 2010    18 46.10     Вінницька
 2010    19 48.14     Вінницька
 2010    20 52.23     Вінницька
 2010    21 55.42     Вінницька
 2010    22 58.27     Вінницька
 2010    23 60.90     Вінницька
 2010    24 62.48     Вінницька
 2010    25 62.71     Вінницька
 2010    26 61.41     Вінницька
 2010    27 62.51     Вінницька
 2010    28 62.21     Вінницька
 2010    29 58.85     Вінницька

In [18]:
# Функція 2: Ряд VHI за діапазон років для вказаних областей
def vhi_by_regions_and_year_range(df, ua_ids, year1, year2):
    """
    Повертає VHI для списку областей за діапазон років
    """
    result = df[(df['ua_id'].isin(ua_ids)) & (df['year'] >= year1) & (df['year'] <= year2)][['year', 'week', 'VHI', 'ua_id', 'province_name']]
    print(f"VHI для областей {ua_ids} за {year1}-{year2} роки:")
    print(result.to_string(index=False))
    return result

# Тест
vhi_by_regions_and_year_range(vhi_df, ua_ids=[1, 2, 3], year1=2000, year2=2002);

VHI для областей [1, 2, 3] за 2000-2002 роки:
 year  week   VHI  ua_id province_name
 2000     1 24.22      1     Вінницька
 2000     2 27.70      1     Вінницька
 2000     3 30.68      1     Вінницька
 2000     4 32.55      1     Вінницька
 2000     5 34.73      1     Вінницька
 2000     6 35.08      1     Вінницька
 2000     7 33.79      1     Вінницька
 2000     8 34.60      1     Вінницька
 2000     9 37.70      1     Вінницька
 2000    10 38.67      1     Вінницька
 2000    11 38.05      1     Вінницька
 2000    12 39.32      1     Вінницька
 2000    13 40.21      1     Вінницька
 2000    14 41.25      1     Вінницька
 2000    15 43.66      1     Вінницька
 2000    16 47.61      1     Вінницька
 2000    17 52.54      1     Вінницька
 2000    18 60.14      1     Вінницька
 2000    19 62.91      1     Вінницька
 2000    20 63.27      1     Вінницька
 2000    21 61.76      1     Вінницька
 2000    22 58.32      1     Вінницька
 2000    23 56.10      1     Вінницька
 2000    24 55.32 

In [19]:
# Функція 3: Екстремуми, середнє та медіана
def vhi_stats(df, ua_ids, year1, year2):
    """
    Повертає min, max, середнє та медіану VHI для вказаних областей та років
    """
    subset = df[(df['ua_id'].isin(ua_ids)) & (df['year'] >= year1) & (df['year'] <= year2)]
    
    stats = subset.groupby(['ua_id', 'province_name'])['VHI'].agg(
        мінімум='min',
        максимум='max',
        середнє='mean',
        медіана='median'
    ).round(2)
    
    print(f"Статистика VHI для областей {ua_ids} за {year1}-{year2} роки:")
    print(stats.to_string())
    return stats

# Тест
vhi_stats(vhi_df, ua_ids=[1, 2, 3], year1=2000, year2=2010);

Статистика VHI для областей [1, 2, 3] за 2000-2010 роки:
                     мінімум  максимум  середнє  медіана
ua_id province_name                                     
1     Вінницька        11.25     81.44    49.52    49.78
2     Волинська        24.65     78.32    52.59    52.12
3     Житомирська      26.51     75.01    51.57    51.88


In [20]:
# Зберігаємо об'єднаний датасет
vhi_df.to_csv('vhi_data/vhi_all_provinces.csv', index=False)
print(f"Датасет збережено: vhi_data/vhi_all_provinces.csv")
print(f"Розмір: {vhi_df.shape}")
print(f"\nПерший та останній рядки:")
print(vhi_df.head(2).to_string())
print(vhi_df.tail(2).to_string())

Датасет збережено: vhi_data/vhi_all_provinces.csv
Розмір: (59022, 10)

Перший та останній рядки:
   year  week    SMN     SMT    VCI    TCI    VHI  province_id province_name  ua_id
0  1982     1  0.068  263.59  63.47  28.34  45.90           24     Вінницька      1
1  1982     2  0.074  265.78  67.62  23.05  45.34           24     Вінницька      1
       year  week    SMN     SMT    VCI   TCI    VHI  province_id province_name  ua_id
59020  2024    51  0.165  278.73  75.79  4.52  40.16            4       АР Крим     27
59021  2024    52  0.164  278.60  76.49  3.42  39.95            4       АР Крим     27
